# RNN Validation Phoneme Analysis

This notebook analyzes a saved RNN run on the validation split. It can either:

- load the already-saved `val_metrics.pkl`
- rerun validation decoding from `best_checkpoint`

It produces:

- trial-level decoded phoneme sequences
- sentence-level PER distribution
- per-phoneme precision / recall / F1 / deletion / insertion summaries
- a phoneme substitution confusion matrix


In [8]:
from pathlib import Path
import sys
import pickle
import shutil
import tempfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import display
from omegaconf import OmegaConf

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "model_training").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from cwd={cwd}. Expected a parent directory containing 'model_training'.")
print(f"Project root: {PROJECT_ROOT}")

MODEL_TRAINING_ROOT = PROJECT_ROOT / "model_training"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(MODEL_TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_TRAINING_ROOT))

from model_training.trainer import BrainToTextDecoder_Trainer
from model_training.helpers.evaluate_model_helpers import LOGIT_TO_PHONEME, _extract_transcription

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("talk")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

PROJECT_ROOT

Project root: /home/alex/snn_for_bts/brain_to_text/model_training/evaluate_model


RuntimeError: Run this notebook from the repo root or from model_training/evaluate_model.

In [ ]:
RUN_DIR = MODEL_TRAINING_ROOT / "logs" / "rnn" / "2026-04-02_17-52-59_035938_rnn_pid238051"
USE_SAVED_VAL_METRICS = True
FORCE_CPU_ON_RERUN = False
KEEP_TEMP_EVAL_DIR = False

VAL_METRICS_PATH = RUN_DIR / "checkpoints" / "val_metrics.pkl"
BEST_CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "best_checkpoint"
TRAINER_CONFIG_PATH = RUN_DIR / "configs" / "trainer_config_resolved.yaml"

assert RUN_DIR.exists(), f"Missing run dir: {RUN_DIR}"
assert TRAINER_CONFIG_PATH.exists(), f"Missing config: {TRAINER_CONFIG_PATH}"

trainer_cfg = OmegaConf.load(TRAINER_CONFIG_PATH)
session_names = list(trainer_cfg.dataset.sessions)

print(f"Run dir: {RUN_DIR}")
print(f"Use saved val metrics: {USE_SAVED_VAL_METRICS}")
print(f"Model type: {trainer_cfg.model_type}")
print(f"Saved avg val PER in config run: best checkpoint under {BEST_CHECKPOINT_PATH.name}")

In [ ]:
def load_saved_val_metrics(run_dir: Path):
    with (run_dir / "checkpoints" / "val_metrics.pkl").open("rb") as f:
        return pickle.load(f)


def rerun_validation(run_dir: Path, force_cpu: bool = False, keep_temp_dir: bool = False):
    cfg = OmegaConf.load(run_dir / "configs" / "trainer_config_resolved.yaml")
    cfg = OmegaConf.create(OmegaConf.to_container(cfg, resolve=True))

    temp_root = Path(tempfile.mkdtemp(prefix="rnn_val_eval_"))
    cfg.mode = "eval"
    if force_cpu:
        cfg.gpu_number = "-1"
    cfg.output_dir = str(temp_root)
    cfg.checkpoint_dir = str(temp_root / "checkpoints")
    cfg.log_dir = str(temp_root / "logs")

    Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
    Path(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)
    Path(cfg.log_dir).mkdir(parents=True, exist_ok=True)

    trainer = BrainToTextDecoder_Trainer(cfg)
    try:
        trainer.load_model_checkpoint(str(run_dir / "checkpoints" / "best_checkpoint"))
        metrics = trainer.validation(
            loader=trainer.val_loader,
            return_logits=True,
            return_data=False,
        )
    finally:
        trainer.close()
        if not keep_temp_dir:
            shutil.rmtree(temp_root, ignore_errors=True)
        else:
            print(f"Kept temp eval dir: {temp_root}")

    return metrics


def ids_to_phonemes(ids):
    return [LOGIT_TO_PHONEME[int(idx)] for idx in ids]


def transcription_array_to_text(arr):
    arr = np.asarray(arr)
    return _extract_transcription(arr)


def edit_distance_only(ref, hyp):
    m, n = len(ref), len(hyp)
    dp = np.zeros((m + 1, n + 1), dtype=np.int32)
    dp[:, 0] = np.arange(m + 1)
    dp[0, :] = np.arange(n + 1)

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            dp[i, j] = min(
                dp[i - 1, j] + 1,
                dp[i, j - 1] + 1,
                dp[i - 1, j - 1] + cost,
            )

    return int(dp[m, n])


def align_sequences(ref, hyp):
    m, n = len(ref), len(hyp)
    dp = np.zeros((m + 1, n + 1), dtype=np.int32)
    dp[:, 0] = np.arange(m + 1)
    dp[0, :] = np.arange(n + 1)

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            dp[i, j] = min(
                dp[i - 1, j] + 1,
                dp[i, j - 1] + 1,
                dp[i - 1, j - 1] + cost,
            )

    i, j = m, n
    aligned = []
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            cost = 0 if ref[i - 1] == hyp[j - 1] else 1
            if dp[i, j] == dp[i - 1, j - 1] + cost:
                op = "match" if cost == 0 else "sub"
                aligned.append((op, ref[i - 1], hyp[j - 1]))
                i -= 1
                j -= 1
                continue
        if i > 0 and dp[i, j] == dp[i - 1, j] + 1:
            aligned.append(("del", ref[i - 1], None))
            i -= 1
            continue
        aligned.append(("ins", None, hyp[j - 1]))
        j -= 1

    aligned.reverse()
    return aligned


if USE_SAVED_VAL_METRICS:
    val_metrics = load_saved_val_metrics(RUN_DIR)
else:
    val_metrics = rerun_validation(RUN_DIR, force_cpu=FORCE_CPU_ON_RERUN, keep_temp_dir=KEEP_TEMP_EVAL_DIR)

print(sorted(val_metrics.keys()))
print("avg_PER:", val_metrics["avg_PER"])
print("avg_loss:", val_metrics["avg_loss"])

In [ ]:
records = []

for batch_idx, decoded_batch in enumerate(val_metrics["decoded_seqs"]):
    true_batch = val_metrics["true_seq"][batch_idx]
    len_batch = val_metrics["phone_seq_lens"][batch_idx]
    transcription_batch = val_metrics["transcription"][batch_idx]
    block_batch = val_metrics["block_nums"][batch_idx]
    trial_batch = val_metrics["trial_nums"][batch_idx]
    day_batch = val_metrics["day_indicies"][batch_idx]

    for item_idx, pred_ids in enumerate(decoded_batch):
        true_len = int(len_batch[item_idx])
        true_ids = np.asarray(true_batch[item_idx][:true_len], dtype=int)
        pred_ids = np.asarray(pred_ids, dtype=int)
        ed = edit_distance_only(true_ids.tolist(), pred_ids.tolist())

        records.append(
            {
                "batch_idx": batch_idx,
                "item_idx": item_idx,
                "day_index": int(day_batch[item_idx]),
                "session": session_names[int(day_batch[item_idx])],
                "block_num": int(block_batch[item_idx]),
                "trial_num": int(trial_batch[item_idx]),
                "transcription": transcription_array_to_text(transcription_batch[item_idx]),
                "true_ids": true_ids.tolist(),
                "pred_ids": pred_ids.tolist(),
                "true_phonemes": ids_to_phonemes(true_ids),
                "pred_phonemes": ids_to_phonemes(pred_ids),
                "true_len": true_len,
                "pred_len": int(len(pred_ids)),
                "edit_distance": ed,
                "sentence_per": ed / true_len if true_len > 0 else np.nan,
            }
        )

trial_df = pd.DataFrame(records)
trial_df = trial_df.sort_values(["session", "block_num", "trial_num"]).reset_index(drop=True)

print(f"Validation trials: {len(trial_df)}")
display(trial_df[["session", "block_num", "trial_num", "transcription", "true_len", "pred_len", "edit_distance", "sentence_per"]].head(10))

In [ ]:
n_classes = len(LOGIT_TO_PHONEME)
class_names = list(LOGIT_TO_PHONEME)

true_count = np.zeros(n_classes, dtype=np.int64)
pred_count = np.zeros(n_classes, dtype=np.int64)
correct_count = np.zeros(n_classes, dtype=np.int64)
deletion_count = np.zeros(n_classes, dtype=np.int64)
insertion_count = np.zeros(n_classes, dtype=np.int64)
substitution_from = np.zeros(n_classes, dtype=np.int64)
substitution_to = np.zeros(n_classes, dtype=np.int64)
confusion = np.zeros((n_classes, n_classes), dtype=np.int64)

for row in trial_df.itertuples(index=False):
    aligned = align_sequences(row.true_ids, row.pred_ids)
    for op, ref_id, hyp_id in aligned:
        if op == "match":
            true_count[ref_id] += 1
            pred_count[hyp_id] += 1
            correct_count[ref_id] += 1
            confusion[ref_id, hyp_id] += 1
        elif op == "sub":
            true_count[ref_id] += 1
            pred_count[hyp_id] += 1
            substitution_from[ref_id] += 1
            substitution_to[hyp_id] += 1
            confusion[ref_id, hyp_id] += 1
        elif op == "del":
            true_count[ref_id] += 1
            deletion_count[ref_id] += 1
        elif op == "ins":
            pred_count[hyp_id] += 1
            insertion_count[hyp_id] += 1

phoneme_df = pd.DataFrame(
    {
        "phoneme": class_names,
        "true_count": true_count,
        "pred_count": pred_count,
        "correct_count": correct_count,
        "deletion_count": deletion_count,
        "insertion_count": insertion_count,
        "substitution_from": substitution_from,
        "substitution_to": substitution_to,
    }
)
phoneme_df["recall"] = np.divide(phoneme_df["correct_count"], phoneme_df["true_count"], out=np.zeros(len(phoneme_df), dtype=float), where=phoneme_df["true_count"] > 0)
phoneme_df["precision"] = np.divide(phoneme_df["correct_count"], phoneme_df["pred_count"], out=np.zeros(len(phoneme_df), dtype=float), where=phoneme_df["pred_count"] > 0)
phoneme_df["f1"] = np.divide(
    2 * phoneme_df["precision"] * phoneme_df["recall"],
    phoneme_df["precision"] + phoneme_df["recall"],
    out=np.zeros(len(phoneme_df), dtype=float),
    where=(phoneme_df["precision"] + phoneme_df["recall"]) > 0,
)
phoneme_df["deletion_rate"] = np.divide(phoneme_df["deletion_count"], phoneme_df["true_count"], out=np.zeros(len(phoneme_df), dtype=float), where=phoneme_df["true_count"] > 0)
phoneme_df["insertion_rate"] = np.divide(phoneme_df["insertion_count"], phoneme_df["pred_count"], out=np.zeros(len(phoneme_df), dtype=float), where=phoneme_df["pred_count"] > 0)

display(phoneme_df.sort_values(["recall", "true_count"], ascending=[False, False]).reset_index(drop=True))

In [ ]:
summary_df = pd.DataFrame(
    {
        "num_trials": [len(trial_df)],
        "avg_sentence_per": [trial_df["sentence_per"].mean()],
        "median_sentence_per": [trial_df["sentence_per"].median()],
        "std_sentence_per": [trial_df["sentence_per"].std()],
        "p90_sentence_per": [trial_df["sentence_per"].quantile(0.90)],
        "p95_sentence_per": [trial_df["sentence_per"].quantile(0.95)],
        "max_sentence_per": [trial_df["sentence_per"].max()],
        "global_per_from_trials": [trial_df["edit_distance"].sum() / trial_df["true_len"].sum()],
        "avg_per_saved_in_metrics": [val_metrics["avg_PER"]],
    }
)
display(summary_df)

session_per_df = (
    trial_df.groupby("session", as_index=False)
    .agg(
        n_trials=("sentence_per", "size"),
        mean_sentence_per=("sentence_per", "mean"),
        median_sentence_per=("sentence_per", "median"),
        total_edit_distance=("edit_distance", "sum"),
        total_true_len=("true_len", "sum"),
    )
)
session_per_df["aggregate_per"] = session_per_df["total_edit_distance"] / session_per_df["total_true_len"]
display(session_per_df.sort_values("aggregate_per", ascending=False).reset_index(drop=True).head(15))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.histplot(trial_df["sentence_per"], bins=40, kde=True, ax=axes[0], color="#1f77b4")
axes[0].set_title("Sentence-Level PER Distribution")
axes[0].set_xlabel("PER")

top_sessions = session_per_df.sort_values("aggregate_per", ascending=False).head(15)
sns.barplot(data=top_sessions, y="session", x="aggregate_per", ax=axes[1], color="#c44e52")
axes[1].set_title("Worst Sessions by Aggregate PER")
axes[1].set_xlabel("Aggregate PER")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

phoneme_plot_df = phoneme_df.query("true_count > 0").sort_values("recall", ascending=False)
sns.barplot(data=phoneme_plot_df, x="phoneme", y="recall", ax=axes[0], color="#4c72b0")
axes[0].tick_params(axis="x", rotation=90)
axes[0].set_title("Per-Phoneme Recall")
axes[0].set_ylim(0, 1.0)

sns.barplot(data=phoneme_plot_df, x="phoneme", y="precision", ax=axes[1], color="#55a868")
axes[1].tick_params(axis="x", rotation=90)
axes[1].set_title("Per-Phoneme Precision")
axes[1].set_ylim(0, 1.0)

plt.tight_layout()
plt.show()

In [ ]:
confusion_df = pd.DataFrame(confusion, index=class_names, columns=class_names)
confusion_no_diag = confusion_df.copy()
np.fill_diagonal(confusion_no_diag.values, 0)

top_confusions = (
    confusion_no_diag.stack()
    .rename("count")
    .reset_index()
    .rename(columns={"level_0": "true_phoneme", "level_1": "pred_phoneme"})
    .query("count > 0")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
display(top_confusions.head(25))

top_true_classes = phoneme_df.query("true_count > 0").sort_values("true_count", ascending=False).head(20)["phoneme"].tolist()
confusion_subset = confusion_no_diag.loc[top_true_classes, top_true_classes]

plt.figure(figsize=(14, 12))
sns.heatmap(confusion_subset, cmap="magma", square=True)
plt.title("Substitution Confusion Matrix (Top 20 Frequent True Phonemes)")
plt.xlabel("Predicted phoneme")
plt.ylabel("True phoneme")
plt.tight_layout()
plt.show()

In [ ]:
worst_sentences = trial_df.sort_values("sentence_per", ascending=False).head(25)
best_sentences = trial_df.sort_values(["sentence_per", "true_len"], ascending=[True, False]).head(25)

display(worst_sentences[["session", "block_num", "trial_num", "transcription", "true_phonemes", "pred_phonemes", "sentence_per"]])
display(best_sentences[["session", "block_num", "trial_num", "transcription", "true_phonemes", "pred_phonemes", "sentence_per"]])

In [ ]:
OUTPUT_DIR = RUN_DIR / "analysis_exports"
OUTPUT_DIR.mkdir(exist_ok=True)

trial_export_cols = [
    "session",
    "block_num",
    "trial_num",
    "transcription",
    "true_len",
    "pred_len",
    "edit_distance",
    "sentence_per",
]

trial_export = trial_df.copy()
trial_export["true_phonemes"] = trial_export["true_phonemes"].apply(lambda x: " ".join(x))
trial_export["pred_phonemes"] = trial_export["pred_phonemes"].apply(lambda x: " ".join(x))

trial_export.to_csv(OUTPUT_DIR / "rnn_val_sentence_level_per.csv", index=False)
phoneme_df.to_csv(OUTPUT_DIR / "rnn_val_phoneme_class_stats.csv", index=False)
top_confusions.to_csv(OUTPUT_DIR / "rnn_val_top_confusions.csv", index=False)
session_per_df.to_csv(OUTPUT_DIR / "rnn_val_session_level_per.csv", index=False)

print(f"Saved exports to: {OUTPUT_DIR}")
display(trial_export[trial_export_cols + ["true_phonemes", "pred_phonemes"]].head(5))